# TSTR ECDF Dataset A - Diabetes

In [1]:
#import libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
print('Libraries imported!!')

Libraries imported!!


In [2]:
#define directory of functions and actual directory
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/chap/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/chap/' #home directory of the project
FUNCTIONS_DIR = 'EVALUATION FUNCTIONS/UTILITY'
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_HOME + FUNCTIONS_DIR)
#import functions for data labelling analisys
from utility_evaluation import DataPreProcessor
from utility_evaluation import train_evaluate_model

#change directory to actual directory
os.chdir(ACTUAL_DIR)
print('Functions imported!!')

Functions imported!!


## 1. Read data

In [3]:
#read real dataset
train_data = pd.read_csv(SYN_DATA_HOME + '1_Chap_Data_Synthetic_ECDF.csv')
categorical_columns = ['group']
for col in categorical_columns :
    train_data[col] = train_data[col].astype('category')
train_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-1.164624,-0.217726,0.659344,0.011437,0.791097,0.658939,0.056327,-0.176931
1,Group0,-2.146172,-0.443908,0.634335,0.253357,0.200205,-0.159470,0.001639,0.344320
2,Group0,0.263697,-1.571178,0.442203,-0.110116,-3.077258,-0.338769,-0.197117,-0.112844
3,Group0,-4.139806,-1.218136,0.593740,0.078172,-4.486668,0.128218,0.104645,-0.004064
4,Group0,4.333972,0.481814,0.261736,0.176678,3.346802,0.090187,-0.027024,-0.010158
...,...,...,...,...,...,...,...,...,...
2712,Group1,-6.930304,1.699628,-0.230742,0.253171,-3.979065,1.555040,0.022290,0.182846
2713,Group1,-4.956003,-0.144191,-0.484976,-0.070694,-6.927887,-0.158235,0.073797,0.087981
2714,Group1,-4.590628,0.733220,-0.964692,-0.067348,-5.401205,0.929108,-0.112441,-0.076244
2715,Group1,-10.420771,0.384690,0.188617,0.104349,-8.064257,2.002390,-0.555722,0.046980


In [4]:
#read test data
test_data = pd.read_csv(REAL_DATA_HOME + '1_Chap_Data_Real_Test.csv')
for col in categorical_columns :
    test_data[col] = test_data[col].astype('category')
test_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-3.053475,-0.177497,-0.010661,0.205118,-2.773102,-0.653440,0.319585,0.017457
1,Group0,-1.375254,0.761393,0.270692,0.140597,-2.157208,-0.297755,0.126143,0.118270
2,Group0,3.681521,1.147414,-0.119372,0.276912,1.452468,0.068735,0.043671,0.074436
3,Group0,0.383070,0.211771,0.374537,0.254718,5.352641,-0.611636,0.344336,0.044627
4,Group0,2.009235,0.765321,0.100557,0.058659,2.842860,0.980271,-0.014527,0.096506
...,...,...,...,...,...,...,...,...,...
674,Group0,1.185955,-0.248240,-0.258972,0.284599,0.816753,0.553623,-0.456024,0.157975
675,Group0,-4.898967,-0.576216,-0.150236,0.069086,-4.450551,-0.192307,0.034382,-0.174072
676,Group0,-3.339095,0.856460,-1.021265,-0.131611,0.639099,0.783467,-0.128578,-0.067689
677,Group0,-3.844234,0.083773,0.334898,-0.210940,-1.259774,0.726944,-0.299066,-0.123455


In [5]:
target = 'group'
#quick look at the breakdown of class values
print('Train data')
print(train_data.shape)
print(train_data.groupby(target).size())
print('#####################################')
print('Test data')
print(test_data.shape)
print(test_data.groupby(target).size())

Train data
(2717, 9)
group
Group0    2606
Group1     111
dtype: int64
#####################################
Test data
(679, 9)
group
Group0    645
Group1     34
dtype: int64


## 2. Pre-process training data

In [6]:
target = 'group'
categorical_columns = []
numerical_columns = train_data.select_dtypes(include=['int64','float64']).columns.tolist()
categories = [np.array(range(2))] if categorical_columns else []
data_preprocessor = DataPreProcessor(categorical_columns, numerical_columns, categories)
x_train = data_preprocessor.preprocess_train_data(train_data.loc[:, train_data.columns != target])
y_train = train_data.loc[:, target]

x_train.shape, y_train.shape

((2717, 8), (2717,))

## 3. Preprocess test data

In [7]:
x_test = data_preprocessor.preprocess_test_data(test_data.loc[:, test_data.columns != target])
y_test = test_data.loc[:, target]
x_test.shape, y_test.shape

((679, 8), (679,))

## 4. Create a dataset to save the results

In [8]:
results = pd.DataFrame(columns = ['model','accuracy','precision','recall','f1'])
results

,model,accuracy,precision,recall,f1


## 4. Train and evaluate Random Forest Classifier

In [9]:
rf_results = train_evaluate_model('RF', x_train, y_train, x_test, y_test)
results = pd.concat([results, rf_results], ignore_index=True)
rf_results

[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.1s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.0s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.0s finished


,model,accuracy,precision,recall,f1
0,RF,0.9529,0.9439,0.9529,0.9348


## 5. Train and Evaluate KNeighbors Classifier

In [10]:
knn_results = train_evaluate_model('KNN', x_train, y_train, x_test, y_test)
results = pd.concat([results, knn_results], ignore_index=True)
knn_results

,model,accuracy,precision,recall,f1
0,KNN,0.9543,0.9564,0.9543,0.9358


## 6. Train and evaluate Decision Tree Classifier

In [11]:
dt_results = train_evaluate_model('DT', x_train, y_train, x_test, y_test)
results = pd.concat([results, dt_results], ignore_index=True)
dt_results

,model,accuracy,precision,recall,f1
0,DT,0.9323,0.9137,0.9323,0.9222


## 7. Train and evaluate Support Vector Machines Classifier

In [12]:
svm_results = train_evaluate_model('SVM', x_train, y_train, x_test, y_test)
results = pd.concat([results, svm_results], ignore_index=True)
svm_results

[LibSVM]WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -515.616153, rho = 0.798196
nSV = 160, nBSV = 0
Total nSV = 160
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -482.981355, rho = 1.588853
nSV = 162, nBSV = 0
Total nSV = 162
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -490.664637, rho = 2.284979
nSV = 165, nBSV = 0
Total nSV = 165
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -507.569475, rho = 0.841547
nSV = 177, nBSV = 0
Total nSV = 177
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -497.164773, rho = 0.930576
nSV = 182, nBSV = 0
Total nSV = 182
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -166.824691, rho = -0.306718
nSV = 93, nBSV = 0
Total nSV = 93


,model,accuracy,precision,recall,f1
0,SVM,0.7216,0.9444,0.7216,0.8001


## 8. Train and evaluate Multilayer Perceptron Classifier

In [13]:
mlp_results = train_evaluate_model('MLP', x_train, y_train, x_test, y_test)
results = pd.concat([results, mlp_results], ignore_index=True)
mlp_results

Iteration 1, loss = 0.54331383
Iteration 2, loss = 0.27648521
Iteration 3, loss = 0.18739526
Iteration 4, loss = 0.16093797
Iteration 5, loss = 0.14502169
Iteration 6, loss = 0.13592314
Iteration 7, loss = 0.12889365
Iteration 8, loss = 0.12184189
Iteration 9, loss = 0.11604403
Iteration 10, loss = 0.11136946
Iteration 11, loss = 0.10604548
Iteration 12, loss = 0.10142498
Iteration 13, loss = 0.09758949
Iteration 14, loss = 0.09446809
Iteration 15, loss = 0.09130522
Iteration 16, loss = 0.08974353
Iteration 17, loss = 0.08601232
Iteration 18, loss = 0.08347638
Iteration 19, loss = 0.08244244
Iteration 20, loss = 0.08063200
Iteration 21, loss = 0.07881570
Iteration 22, loss = 0.07788688
Iteration 23, loss = 0.08020412
Iteration 24, loss = 0.07556077
Iteration 25, loss = 0.07353745
Iteration 26, loss = 0.07278760
Iteration 27, loss = 0.07114270
Iteration 28, loss = 0.06891085
Iteration 29, loss = 0.07012469
Iteration 30, loss = 0.06728460
Iteration 31, loss = 0.06597763
Iteration 32, los

,model,accuracy,precision,recall,f1
0,MLP,0.9367,0.9261,0.9367,0.9308


## 9. Save results file

In [14]:
results.to_csv('RESULTS/models_results_ECDF.csv', index=False)
results

,model,accuracy,precision,recall,f1
0,RF,0.9529,0.9439,0.9529,0.9348
1,KNN,0.9543,0.9564,0.9543,0.9358
2,DT,0.9323,0.9137,0.9323,0.9222
3,SVM,0.7216,0.9444,0.7216,0.8001
4,MLP,0.9367,0.9261,0.9367,0.9308
